# Figure 10: Conflict Impact Simulation Analysis

## Purpose
Conduct **sensitivity analysis** to assess how changes in conflict variables affect food crisis phase predictions.

## Methodology
1. **Baseline Model**: Train forecasting/nowcasting models on full data
2. **Conflict Multiplier**: Systematically increase/decrease conflict features (fatalities, events)
3. **Phase Transitions**: Track how predictions change with varying conflict intensities
4. **Heatmap Visualization**: Create matrix showing phase transitions under different conflict scenarios

## Key Questions
- How sensitive are predictions to conflict escalation?
- Which conflict types (battles, explosions, violence) have strongest impact?
- What is the threshold for conflict-induced phase changes?

## Expected Outputs
- Heatmap of phase transitions vs. conflict multiplier
- Quantification of conflict feature importance
- Policy-relevant insights on conflict-food security linkages

## Note on Paths
**Execution note**: The released notebook uses package-relative paths for source data and generated outputs.

In [ ]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from food_crisis_functions import *
import json

with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)
    
with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)


# read csv
df = pd.read_csv(r'../1.Source Data/Forecasting_Analysis_010825.csv')



###drop fews_ipc_ha
#df = df.drop(['fews_ipc_ha'], axis=1)
# random split train and test
df_origin = df.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#drop overall phase
df = df.drop(['overall_phase'], axis=1)
#for each region, set last observation to be test set
# create a series of new outcome, phase2_worse=phase2_percent+phase3_percent+phase4_percent+phase5_percent, phase3_worse=phase3_percent+phase4_percent+phase5_percent, phase4_worse=phase4_percent+phase5_percent, phase5_worse=phase5_percent
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
#drop phase2_percent, phase3_percent, phase4_percent, phase5_percent, phase1_percent
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)
# Splitting the data
#test_df = df.groupby('area_id').tail(1)
#train_df = df.drop(test_df.index)
#test_df = test_df.drop(['area_id','date'], axis=1)
#train_df = train_df.drop(['area_id','date'], axis=1)
y_pred_test = pd.DataFrame()

conflict_vars = ['fatalities_battles_l12',
 'fatalities_explosions_l12',
 'fatalities_violence_l12',
 'event_count_battles_l12',
 'event_count_explosions_l12',
 'event_count_violence_l12',
 'fatalities_battles_w5_l12',
 'fatalities_explosions_w5_l12',
 'fatalities_violence_w5_l12',
 'event_count_battles_w5_l12',
 'event_count_explosions_w5_l12',
 'event_count_violence_w5_l12',
 'fatalities_battles_w10_l12',
 'fatalities_explosions_w10_l12',
 'fatalities_violence_w10_l12',
 'event_count_battles_w10_l12',
 'event_count_explosions_w10_l12',
 'event_count_violence_w10_l12']
shape_values_df_ensemble = pd.DataFrame()
df_result = pd.DataFrame()
date = "2022-01-01"  # Define the 'date' variable
#order unique_dates
y_pred_test=pd.DataFrame()
for i in range(2, 6):
    train_df = df[df['date'] < date]
    test_df = df[df['date'] >= date]
    train_df['month'] = pd.to_datetime(train_df['date']).dt.month
    test_df['month'] = pd.to_datetime(test_df['date']).dt.month
    train_df['year'] = pd.to_datetime(train_df['date']).dt.year
    test_df['year'] = pd.to_datetime(test_df['date']).dt.year
    train_df = train_df.drop(['date'], axis=1)
    test_df = test_df.drop(['date'], axis=1)
    train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_df_new = test_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_index = test_df_new.index
    X_train = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train = train_df_new['phase{}_worse'.format(i)]
    X_pred = test_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_pred = test_df_new['phase{}_worse'.format(i)]
    X__pred = X_pred.copy()
    X__train = X_train.copy()
    y__train = y_train.copy()
    y__pred = y_pred.copy()
    X_pred_loc = X_pred[['lat', 'lon', 'month','year','area_id']]
    X_pred_loc = pd.concat([X_pred_loc, pd.get_dummies(X_pred_loc['month'], prefix='month'),pd.get_dummies(X_pred_loc['area_id'],prefix="area")], axis=1)
    for k in range(1, 13):
        if 'month_{}'.format(k) not in X_pred_loc.columns:
            X_pred_loc['month_{}'.format(k)] = False
    for k in range(1, 1876):
        if 'area_{}'.format(k) not in X_pred_loc.columns:
            X_pred_loc['area_{}'.format(k)] = False
    X_pred_loc = X_pred_loc.drop(['month','area_id'], axis=1)
    X_stable = X_pred[['elevation','market_access', 'nitrogen_5-15cm_mean',
    'phh2o_5-15cm_mean',
    'cec_5-15cm_mean',
    'cfvo_5-15cm_mean',
    'soc_5-15cm_mean','aez_groupid_4000',
    'aez_groupid_7000',
    'aez_groupid_9000',
    'aez_groupid_10000',
    'aez_groupid_12000',
    'aez_groupid_17000',
    'aez_groupid_19000',
    'aez_groupid_25000',
    'aez_groupid_30000',
    'aez_groupid_31000',
    'aez_groupid_32000',
    'aez_groupid_33000',
    'aez_groupid_34000',
    'aez_groupid_36000',
    'aez_groupid_40000',
    'aez_groupid_43000',
    'slope',
    'estimated_population',
    'cropland',
    'rangeland','area',
    'es_urban_pop',
    'urban_area',
    'distance_to_river',
    'ruggedness_index']]
    X_pred = X_pred.drop(['lat', 'lon', 'month','year','area_id','elevation','market_access','nitrogen_5-15cm_mean',
    'phh2o_5-15cm_mean',
    'cec_5-15cm_mean',
    'cfvo_5-15cm_mean',
    'soc_5-15cm_mean','aez_groupid_4000',
    'aez_groupid_7000',
    'aez_groupid_9000',
    'aez_groupid_10000',
    'aez_groupid_12000',
    'aez_groupid_17000',
    'aez_groupid_19000',
    'aez_groupid_25000',
    'aez_groupid_30000',
    'aez_groupid_31000',
    'aez_groupid_32000',
    'aez_groupid_33000',
    'aez_groupid_34000',
    'aez_groupid_36000',
    'aez_groupid_40000',
    'aez_groupid_43000',
    'slope',
    'estimated_population',
    'cropland',
    'rangeland','area',
    'es_urban_pop',
    'urban_area',
    'distance_to_river',
    'ruggedness_index'], axis=1)
    X_pred = pd.DataFrame(np.nan, index=X_pred.index, columns=X_pred.columns)
    X_pred = pd.concat([X_pred_loc,X_stable,X_pred], axis=1)
    X_pred = X_pred.loc[:,~X_pred.columns.duplicated()]
    X_pred = X__pred.drop([ 'month','year'], axis=1)
    #X_pred = X_pred.fillna(0)
    y_test = test_df_new['phase{}_worse'.format(i)]
    fews_ipc_ha_test = X_pred['fews_ipc_ha']

    X_train_loc = X_train[['lat', 'lon', 'month','year','area_id']]
    X_train_loc = pd.concat([X_train_loc, pd.get_dummies(X_train_loc['month'], prefix='month'),pd.get_dummies(X_train_loc['area_id'],prefix="area")], axis=1)
    for k in range(1, 13):
        if 'month_{}'.format(k) not in X_train_loc.columns:
            X_train_loc['month_{}'.format(k)] = False
    for k in range(1, 1876):
        if 'area_{}'.format(k) not in X_train_loc.columns:
            X_train_loc['area_{}'.format(k)] = False
    X_train_loc = X_train_loc.drop(['month','area_id'], axis=1)
    X_train = X_train.drop(['lat', 'lon', 'month','year','area_id'], axis=1)
    X_train = pd.concat([X_train_loc, X_train], axis=1)
    X_train = X_train.loc[:,~X_train.columns.duplicated()]
    X_train = X__train.drop([ 'month','year'], axis=1)
    #X_train = X_train.drop(['lat', 'lon', 'month','year'], axis=1)
    X_train = X_train[X_pred.columns]
    if i == 3:
        best_params_xgb_regressor = best_params_xgb_regressor_for_p3

    model = xgb.XGBRegressor(**best_params_xgb_regressor)
    #X_train = X_train.drop(['fews_ipc_ha'], axis=1)
    #X_pred = X_pred.drop(['fews_ipc_ha'], axis=1)
    model.fit(X_train, y_train)
    # Predictions
    y_pred = model.predict(X_pred)
    #y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'fews_ipc_ha':fews_ipc_ha_test,'test_index':test_index})], ignore_index=True)
    y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'test_index':test_index})], ignore_index=True)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train)
    shap_values_df = pd.DataFrame(shap_values, columns=X_train.columns)
    shap_values_df['phase'] = i
    shape_values_df_ensemble = pd.concat([shape_values_df_ensemble, shap_values_df], ignore_index=True)
   
y_pred_test = convert_prob_to_phase(y_pred_test)
y_pred_old = y_pred_test['overall_phase_pred']

shape_values_df_ensemble = pd.DataFrame()
df_result = pd.DataFrame()
date = "2022-01-01"  # Define the 'date' variable
#order unique_dates
y_pred_test=pd.DataFrame()
for i in range(2, 6):
    train_df = df[df['date'] < date]
    test_df = df[df['date'] >= date]
    train_df['month'] = pd.to_datetime(train_df['date']).dt.month
    test_df['month'] = pd.to_datetime(test_df['date']).dt.month
    train_df['year'] = pd.to_datetime(train_df['date']).dt.year
    test_df['year'] = pd.to_datetime(test_df['date']).dt.year
    train_df = train_df.drop(['date'], axis=1)
    test_df = test_df.drop(['date'], axis=1)
    train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_df_new = test_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_index = test_df_new.index
    X_train = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train = train_df_new['phase{}_worse'.format(i)]
    X_pred = test_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_pred = test_df_new['phase{}_worse'.format(i)]
    X__pred = X_pred.copy()
    X__train = X_train.copy()
    y__train = y_train.copy()
    y__pred = y_pred.copy()
    X_pred_loc = X_pred[['lat', 'lon', 'month','year','area_id']]
    X_pred_loc = pd.concat([X_pred_loc, pd.get_dummies(X_pred_loc['month'], prefix='month'),pd.get_dummies(X_pred_loc['area_id'],prefix="area")], axis=1)
    for k in range(1, 13):
        if 'month_{}'.format(k) not in X_pred_loc.columns:
            X_pred_loc['month_{}'.format(k)] = False
    for k in range(1, 1876):
        if 'area_{}'.format(k) not in X_pred_loc.columns:
            X_pred_loc['area_{}'.format(k)] = False
    X_pred_loc = X_pred_loc.drop(['month','area_id'], axis=1)
    X_stable = X_pred[['elevation','market_access', 'nitrogen_5-15cm_mean',
    'phh2o_5-15cm_mean',
    'cec_5-15cm_mean',
    'cfvo_5-15cm_mean',
    'soc_5-15cm_mean','aez_groupid_4000',
    'aez_groupid_7000',
    'aez_groupid_9000',
    'aez_groupid_10000',
    'aez_groupid_12000',
    'aez_groupid_17000',
    'aez_groupid_19000',
    'aez_groupid_25000',
    'aez_groupid_30000',
    'aez_groupid_31000',
    'aez_groupid_32000',
    'aez_groupid_33000',
    'aez_groupid_34000',
    'aez_groupid_36000',
    'aez_groupid_40000',
    'aez_groupid_43000',
    'slope',
    'estimated_population',
    'cropland',
    'rangeland','area',
    'es_urban_pop',
    'urban_area',
    'distance_to_river',
    'ruggedness_index']]
    X_pred = X_pred.drop(['lat', 'lon', 'month','year','area_id','elevation','market_access','nitrogen_5-15cm_mean',
    'phh2o_5-15cm_mean',
    'cec_5-15cm_mean',
    'cfvo_5-15cm_mean',
    'soc_5-15cm_mean','aez_groupid_4000',
    'aez_groupid_7000',
    'aez_groupid_9000',
    'aez_groupid_10000',
    'aez_groupid_12000',
    'aez_groupid_17000',
    'aez_groupid_19000',
    'aez_groupid_25000',
    'aez_groupid_30000',
    'aez_groupid_31000',
    'aez_groupid_32000',
    'aez_groupid_33000',
    'aez_groupid_34000',
    'aez_groupid_36000',
    'aez_groupid_40000',
    'aez_groupid_43000',
    'slope',
    'estimated_population',
    'cropland',
    'rangeland','area',
    'es_urban_pop',
    'urban_area',
    'distance_to_river',
    'ruggedness_index'], axis=1)
    X_pred = pd.DataFrame(np.nan, index=X_pred.index, columns=X_pred.columns)
    X_pred = pd.concat([X_pred_loc,X_stable,X_pred], axis=1)
    X_pred = X_pred.loc[:,~X_pred.columns.duplicated()]
    X_pred = X__pred.drop([ 'month','year'], axis=1)
    #X_pred = X_pred.fillna(0)
    y_test = test_df_new['phase{}_worse'.format(i)]
    fews_ipc_ha_test = X_pred['fews_ipc_ha']

    X_train_loc = X_train[['lat', 'lon', 'month','year','area_id']]
    X_train_loc = pd.concat([X_train_loc, pd.get_dummies(X_train_loc['month'], prefix='month'),pd.get_dummies(X_train_loc['area_id'],prefix="area")], axis=1)
    for k in range(1, 13):
        if 'month_{}'.format(k) not in X_train_loc.columns:
            X_train_loc['month_{}'.format(k)] = False
    for k in range(1, 1876):
        if 'area_{}'.format(k) not in X_train_loc.columns:
            X_train_loc['area_{}'.format(k)] = False
    X_train_loc = X_train_loc.drop(['month','area_id'], axis=1)
    X_train = X_train.drop(['lat', 'lon', 'month','year','area_id'], axis=1)
    X_train = pd.concat([X_train_loc, X_train], axis=1)
    X_train = X_train.loc[:,~X_train.columns.duplicated()]
    X_train = X__train.drop([ 'month','year'], axis=1)
    #X_train = X_train.drop(['lat', 'lon', 'month','year'], axis=1)
    X_train = X_train[X_pred.columns]
    if i == 3:
        best_params_xgb_regressor = best_params_xgb_regressor_for_p3
    
    #all conflict variables in test data 100 percent
    X_pred[conflict_vars] = X_pred[conflict_vars]*1.1
    
    model = xgb.XGBRegressor(**best_params_xgb_regressor)
    model.fit(X_train, y_train)
    # Predictions
    y_pred = model.predict(X_pred)
    # for y_pred_test, add a column to indicate the phase
    y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'fews_ipc_ha':fews_ipc_ha_test,'test_index':test_index})], ignore_index=True)
    #y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'test_index':test_index})], ignore_index=True)
   
y_pred_test = convert_prob_to_phase(y_pred_test)
y_pred_original = y_pred_test['overall_phase']
y_pred_new_10 = y_pred_test['overall_phase_pred']
# combine y_pred_old and y_pred_new
y_pred_frame = pd.DataFrame({'y_pred_original':y_pred_original,'y_pred_old': y_pred_old, 'y_pred_new_10': y_pred_new_10})
# summarize all the flows from y_pred_old to y_pred_new
# create a dataframe to store the flow
flow_df = pd.DataFrame(columns=['from', 'to', 'count'])
# create a dataframe to store the flow
for i in range(1, 6):
    for j in range(1, 6):
        flow_df = pd.concat([flow_df, pd.DataFrame({'from': [i], 'to': [j], 'count': [len(y_pred_frame[(y_pred_frame['y_pred_original']==i) & (y_pred_frame['y_pred_new_10']==j)])]})], ignore_index=True)
        
# create a matrix to store the flow
flow_matrix = np.zeros((5, 5))

# fill in the flow_matrix
for i in range(1, 6):
    for j in range(1, 6):
        flow_matrix[i-1, j-1] = len(y_pred_frame[(y_pred_frame['y_pred_original']==i) & (y_pred_frame['y_pred_new_10']==j)])
        
# plot the flow_matrix
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(flow_matrix, annot=True, fmt='g', cmap='Blues', ax=ax)
ax.set_xlabel('up_10')
ax.set_ylabel('real_level')
ax.set_title('flow matrix')
#set ticks
ax.set_xticklabels([1, 2, 3, 4, 5])
ax.set_yticklabels([1, 2, 3, 4, 5])
plt.savefig(
    'produced_graph/Picture5.jpg', 
    dpi=300,
    format='jpeg',
    bbox_inches='tight',
    pil_kwargs={'compression': 'lzw'}
)
plt.show()